In [115]:
import torch
import numpy as np

from sklearn.datasets import load_wine
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, MSELoss, CrossEntropyLoss
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam

In [116]:
class MyDataset(Dataset):
    def __init__(self, datas: np.ndarray, labels: np.ndarray) -> None:
        self._x = torch.from_numpy(datas).float()
        self._y = torch.from_numpy(labels).long()

    def __len__(self) -> int:
        return len(self._x)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        int(idx)
        return self._x[idx], self._y[idx]

In [117]:
def train_model_reg(X, Y, model, skf, batch, epochs, optmizer, criterion):
    f1_scores = []
    accuracy_scores = []
    for i, (train_index, test_index) in enumerate(skf.split(X, Y)):
        print(f"Treinando modelo ----- Fold {i + 1}")
        train_data = MyDataset(X[train_index], Y[train_index])
        train_loader = DataLoader(train_data, batch_size=batch)
        for i_epochs in range(epochs):    
            model.train()
            for x, y in train_loader:
                optmizer.zero_grad()
                output = model(x).cpu()
                loss = criterion(output, y)
                loss.backward()
                optmizer.step()

        model.eval()
        with torch.no_grad():
            x_test_tensor = Tensor(X[test_index]).float()
            output = model(x_test_tensor).cpu()
            result = torch.argmax(output, dim=1).cpu().numpy()
            f1_s = f1_score(Y[test_index], result, average="macro")
            accuracy_s = accuracy_score(Y[test_index], result)
            f1_scores.append(f1_s)
            accuracy_scores.append(accuracy_s)
            print(f"f1: {f1_s}")
            print(f"Accuracy: {accuracy_s}")

    print(f"\nResultado modelo ----- F1: {np.mean(f1_scores)},  accuracy:{np.mean(accuracy_scores)}")

In [118]:
dataset = load_wine()
X = dataset.data
Y = dataset.target

In [119]:
print(X.shape)

(178, 13)


In [120]:
model = Sequential(
    Linear(13, 32),
    ReLU(),
    Linear(32,16),
    ReLU(),
    Linear(16, 3)
)

In [121]:
optmizer = Adam(model.parameters(), lr=0.001)
criterion = CrossEntropyLoss()
max_epochs = 500
batch = 12
skf = StratifiedKFold(n_splits=5, shuffle=True,  random_state=42)

In [122]:
train_model_reg(X, Y, model, skf, batch, max_epochs, optmizer, criterion)

Treinando modelo ----- Fold 1
f1: 0.945824706694272
Accuracy: 0.9444444444444444
Treinando modelo ----- Fold 2
f1: 0.9740129935032483
Accuracy: 0.9722222222222222
Treinando modelo ----- Fold 3
f1: 0.9457411645054665
Accuracy: 0.9444444444444444
Treinando modelo ----- Fold 4
f1: 0.9740129935032483
Accuracy: 0.9714285714285714
Treinando modelo ----- Fold 5
f1: 1.0
Accuracy: 1.0

Resultado modelo ----- F1: 0.967918371641247,  accuracy:0.9665079365079364
